# 🔬 Notebook 3: Gmail — Deep Dive

Three deep dives where most real-world Gmail engineering time is spent:

1. **SMTP** — the wire protocol that glues the world's mail servers together.
2. **Search** — going from "scan every message" to a real inverted index.
3. **Storage tiers + bounces** — keeping costs under control and handling failure.

## 🛠️ Setup

```bash
cd 06-system-designs/gmail
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## Deep dive 1 — SMTP

When our outbound relay sends mail to `alice@example.com`:

```
1. DNS query: MX records for example.com → mx1.example.com (priority 10)
2. TCP connect to mx1.example.com:25
3. SMTP conversation:
   S: 220 mx1.example.com ESMTP ready
   C: EHLO our.server.com
   S: 250-mx1.example.com ... 250 OK
   C: MAIL FROM:<us@ourdomain.com>
   S: 250 OK
   C: RCPT TO:<alice@example.com>
   S: 250 OK
   C: DATA
   S: 354 Start mail input
   C: <headers + blank line + body>
   C: .                      ← single dot on its own line terminates DATA
   S: 250 Queued as ABC123
   C: QUIT
```

`MAIL FROM` is the **envelope sender** — where bounces go. `RCPT TO` is the
envelope recipient. A single message can have multiple `RCPT TO` commands; the
receiving server fan-outs to each local user on its side.

In [1]:
# A small SMTP-like state machine.  DATA ends on a line containing only ".".
class MiniSmtp:
    def __init__(self):
        self.state = "INIT"
        self.mail_from = None
        self.rcpts: list[str] = []
        self.data_lines: list[str] = []

    def handle(self, line: str) -> str:
        u = line.upper()
        # DATA body mode: collect until single "." terminator
        if self.state == "DATA":
            if line == ".":
                self.state = "DONE"
                return "250 Queued as MSG-001"
            self.data_lines.append(line)
            return ""                      # no response mid-DATA
        if self.state == "INIT" and u.startswith("EHLO"):
            self.state = "GREETED"; return "250 hello"
        if self.state in ("GREETED", "DONE") and u.startswith("MAIL FROM:"):
            self.mail_from = line.split(":", 1)[1].strip()
            self.rcpts = []; self.data_lines = []
            self.state = "GOT_SENDER"; return "250 OK"
        if self.state in ("GOT_SENDER", "GOT_RCPT") and u.startswith("RCPT TO:"):
            self.rcpts.append(line.split(":", 1)[1].strip())
            self.state = "GOT_RCPT"; return "250 OK"
        if self.state == "GOT_RCPT" and u == "DATA":
            self.state = "DATA"; return "354 end data with <CRLF>.<CRLF>"
        if u == "QUIT":
            self.state = "CLOSED"; return "221 bye"
        return "500 bad sequence"

s = MiniSmtp()
script = [
    "EHLO me.com",
    "MAIL FROM:<a@me.com>",
    "RCPT TO:<b@you.com>",
    "RCPT TO:<c@you.com>",
    "DATA",
    "Subject: Hi",
    "",
    "Hello there!",
    ".",
    "QUIT",
]
for cmd in script:
    resp = s.handle(cmd)
    print(f"C> {cmd}")
    if resp:
        print(f"S> {resp}")
print("\nenvelope:", s.mail_from, "→", s.rcpts)
print("body lines:", s.data_lines)

C> EHLO me.com
S> 250 hello
C> MAIL FROM:<a@me.com>
S> 250 OK
C> RCPT TO:<b@you.com>
S> 250 OK
C> RCPT TO:<c@you.com>
S> 250 OK
C> DATA
S> 354 end data with <CRLF>.<CRLF>
C> Subject: Hi
C> 
C> Hello there!
C> .
S> 250 Queued as MSG-001
C> QUIT
S> 221 bye

envelope: <a@me.com> → ['<b@you.com>', '<c@you.com>']
body lines: ['Subject: Hi', '', 'Hello there!']


### Retries and bounces

Receiving servers sometimes respond with `4xx` (temporary — retry later) or `5xx`
(permanent — stop trying). A robust relay uses **exponential backoff with jitter**:
it tries at 1 min, 5 min, 15 min, 1 h, 4 h… for up to 3 days, then gives up and
generates a **bounce** message (a new email *from* `MAILER-DAEMON@ourdomain.com`
to the original sender).

In [2]:
# Simulate a retry schedule for a flaky recipient.
import random

def retry_schedule(max_age_hours=72):
    delay_s = 60          # start at 1 minute
    elapsed = 0
    while elapsed < max_age_hours * 3600:
        jitter = random.uniform(0.8, 1.2)
        yield int(elapsed), int(delay_s * jitter)
        elapsed += delay_s * jitter
        delay_s = min(delay_s * 4, 4 * 3600)    # cap at 4h between tries

random.seed(0)
print(f"{'elapsed(s)':>10}  {'next delay(s)':>14}")
for i, (t, d) in enumerate(retry_schedule()):
    print(f"{t:>10}  {d:>14}")
    if i > 8: break

elapsed(s)   next delay(s)
         0              68
        68             264
       333             929
      1262            3469
      4732           14464
     19197           13852
     33049           16034
     49084           13267
     62351           14265
     76616           14880


## Deep dive 2 — Full-text search

A mail search engine must handle queries like:

```
from:alice subject:report attachment:yes after:2025/01/01 quarterly
```

It's a mix of **text search** (the word *quarterly*), **structured filters**
(`from:`, `attachment:`) and **ranges** (`after:`).

### v1 — linear scan (BAD)

Just loop through messages and look for the word. O(N) per query, fine for 10 emails,
fatal for 10 million.

In [3]:
# Naive scan.
import time

messages = [
    {"from": "alice@ex.com", "subject": "Q3 report",  "body": "see attached",      "has_attach": True},
    {"from": "bob@ex.com",   "subject": "Lunch",      "body": "q3 discussion",     "has_attach": False},
    {"from": "alice@ex.com", "subject": "Random",     "body": "hello"},
] * 50_000    # 150k messages

def naive_search(term):
    hits = []
    for m in messages:
        if term in m["subject"].lower() or term in m["body"].lower():
            hits.append(m)
    return hits

t = time.perf_counter()
hits = naive_search("quarterly")
print(f"naive scan over {len(messages):>7} msgs: {(time.perf_counter()-t)*1000:.1f} ms  → {len(hits)} hits")

naive scan over  150000 msgs: 13.8 ms  → 0 hits


### v2 — inverted index (BETTER)

An inverted index maps each **term → set of message IDs**. Looking up a word is O(1);
combining words is set intersection. This is the core of Lucene / Elasticsearch.

In [4]:
import re, time
from collections import defaultdict

class Inverted:
    def __init__(self):
        self.msgs: list[dict] = []
        self.term_idx: dict[str, set[int]] = defaultdict(set)
        # structured fields
        self.by_from: dict[str, set[int]] = defaultdict(set)
        self.has_attach: set[int] = set()

    def add(self, m):
        mid = len(self.msgs); self.msgs.append(m)
        for w in re.findall(r"\w+", (m["subject"] + " " + m["body"]).lower()):
            self.term_idx[w].add(mid)
        self.by_from[m["from"].lower()].add(mid)
        if m.get("has_attach"):
            self.has_attach.add(mid)

    def search(self, query: str):
        candidates: set[int] | None = None
        def narrow(s): return s if candidates is None else candidates & s

        for tok in query.lower().split():
            if tok.startswith("from:"):
                candidates = narrow(self.by_from.get(tok[5:], set()))
            elif tok == "has:attachment":
                candidates = narrow(self.has_attach)
            else:
                candidates = narrow(self.term_idx.get(tok, set()))
            if not candidates:
                return []
        return [self.msgs[i] for i in (candidates or set())]

idx = Inverted()
for m in messages:
    idx.add(m)

t = time.perf_counter()
r = idx.search("q3 has:attachment from:alice@ex.com")
print(f"indexed search: {(time.perf_counter()-t)*1000:.2f} ms  → {len(r)} hits")

indexed search: 1.31 ms  → 50000 hits


### Why **per-user** indices?

In real Gmail, each user's mail is indexed into its own logical shard:
- **Privacy** — one bug can't leak another user's data.
- **Size** — most users have < 50 k messages, so per-user indices fit in memory.
- **Cost** — deleting a user is just "drop this shard."

### Indexing is async
The mail-write path **does not** write to the index synchronously. Instead it emits
a new-message event (Kafka/PubSub), and an indexer consumes it. Users typically see
new mail in search within a few seconds — a trade-off we're happy to make because
blocking SMTP on search indexing would cause bounces when the index is slow.

## Deep dive 3 — Storage tiers and attachment dedup

Inbox access is extremely **skewed**: the newest ~5% of messages are read 90% of
the time. We exploit this with tiered storage.

```
   hot :  SSD + in-memory cache  (last 30 days)
   warm:  HDD / standard object store  (30 days – 1 year)
   cold:  archive storage (Glacier-class) (> 1 year)
```

Cold reads take seconds, but that's fine for rare "I need an email from 2019"
lookups. The savings are dramatic — often >10× at scale.

In [5]:
# Simulate the cost of keeping everything hot vs. tiering.
from datetime import datetime, timedelta, timezone
import random

random.seed(42)
now = datetime.now(timezone.utc)

# generate 100k messages with realistic age distribution
messages = []
for _ in range(100_000):
    # heavy tail: most messages are recent, few are ancient
    days_old = int(random.expovariate(1/180))      # mean 180 days
    messages.append({
        "received_at": now - timedelta(days=days_old),
        "size_kb": random.randint(20, 200),
    })

COST_PER_GB_MONTH = {"hot": 0.20, "warm": 0.04, "cold": 0.004}

def tier_of(age_days):
    if age_days < 30:  return "hot"
    if age_days < 365: return "warm"
    return "cold"

tier_bytes = {"hot": 0, "warm": 0, "cold": 0}
for m in messages:
    age = (now - m["received_at"]).days
    tier_bytes[tier_of(age)] += m["size_kb"] * 1024

total_gb = sum(tier_bytes.values()) / (1024**3)
tiered_cost  = sum(b / (1024**3) * COST_PER_GB_MONTH[t] for t, b in tier_bytes.items())
all_hot_cost = total_gb * COST_PER_GB_MONTH["hot"]

print(f"total mail: {total_gb:.3f} GB")
for t, b in tier_bytes.items():
    print(f"  {t:<5} {b/(1024**3):7.3f} GB  (${b/(1024**3)*COST_PER_GB_MONTH[t]:.4f}/mo)")
print(f"\nmonthly storage cost — all-hot: ${all_hot_cost:.3f}")
print(f"monthly storage cost — tiered : ${tiered_cost:.3f}")
print(f"savings                         : {(1 - tiered_cost/all_hot_cost)*100:.1f}%")

total mail: 10.511 GB
  hot     1.617 GB  ($0.3233/mo)
  warm    7.531 GB  ($0.3013/mo)
  cold    1.363 GB  ($0.0055/mo)

monthly storage cost — all-hot: $2.102
monthly storage cost — tiered : $0.630
savings                         : 70.0%


### Attachment deduplication

When someone forwards a 5 MB PDF to a mailing list of 500 people, do we store 500
copies? No — we **content-hash** the attachment (SHA-256) and store it once in the
object store. Each message row references the hash.

In [6]:
import hashlib

class AttachmentStore:
    def __init__(self):
        self.blobs: dict[str, bytes] = {}   # sha256 → bytes (pretend object store)
        self.refs:  dict[str, int]   = {}   # sha256 → refcount

    def put(self, data: bytes) -> str:
        h = hashlib.sha256(data).hexdigest()
        if h not in self.blobs:
            self.blobs[h] = data
            self.refs[h] = 0
        self.refs[h] += 1
        return h

    def delete(self, h: str) -> None:
        self.refs[h] -= 1
        if self.refs[h] == 0:
            del self.blobs[h]; del self.refs[h]

    def stats(self):
        return {"unique_blobs": len(self.blobs), "total_refs": sum(self.refs.values()),
                "bytes_stored": sum(len(b) for b in self.blobs.values())}

store = AttachmentStore()
pdf = b"PDF-fake-bytes" * 1000             # ~14 KB
# 500 recipients each get the same PDF attached
for _ in range(500):
    store.put(pdf)

print(store.stats())        # unique_blobs=1, total_refs=500 — huge win!

{'unique_blobs': 1, 'total_refs': 500, 'bytes_stored': 14000}


## 🎯 Wrap-up

We started with a naive single-server inbox and layered on:

- A **queue** between SMTP and application writes (durability).
- **Partitioning** metadata by `user_id` (scale).
- **Object storage** for bodies and attachments (cheap).
- **Content-hash deduplication** for attachments (cheaper).
- A per-user **inverted index** populated asynchronously (search).
- **Hot / warm / cold** tiers with an age-based migrator (cost).
- **Exponential-backoff retries** with DSN/bounce on permanent failure (correctness).

Each of these came from an actual, measurable problem with the previous version —
which is the *point* of the bad → best progression: never add a box until you
feel the pain of its absence.

### Further reading
- **RFC 5321** — SMTP
- **RFC 5322** — Internet message format (headers, threading)
- **RFC 2045–2049** — MIME
- **RFC 7208 / 6376 / 7489** — SPF / DKIM / DMARC
- **Lucene** docs on inverted indices
- Google SRE book, chapter on Gmail capacity planning.